# Astra — word-level English base + chat fine-tune on GPU

Trains the 8.1M-param word-level model on a free T4 GPU in roughly 15-25 min:
1. **word-prose** 5400 steps (English base, val ppl ~56)
2. **word-chat** 3800 steps warm-started from prose final (chat fine-tune)

Checkpoints are saved in the **same `.npz` format** as the repo's NumPy inference path, so the downloaded `final.npz` files drop straight into `checkpoints/` and work with `make chat`.

In [0]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

**Runtime → Change runtime type → T4 GPU** if the cell above shows `cuda available: False`. Require restart after switching.

In [0]:
# Clone the repo + install deps
import os
REPO = '/content/astra'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}
os.chdir(REPO)
!pip install -q numpy  # torch is preinstalled on Colab
print('cwd:', os.getcwd())

**Build the word tokenizer artifact** (gitignored in the repo, so it's generated here from the corpora):

In [0]:
%cd /content/astra
!python tools/build_word_tokenizer.py
import json
a = json.load(open('/content/astra/tokenizer/artifacts/prose_chat_word.json'))
print('tokenizer vocab:', a['vocab_size'])


**Smoke test** — 10 steps of word-prose to verify the pipeline on the T4 before the real run:

In [0]:
%cd /content/astra
!mkdir -p /content/runs
!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 10 --out /content/runs/word_prose_smoke \
  --cache-dir /content/cache 2>&1 | tail -8


Sanity: last line should show `[torch] device=cuda`, a `[step ...]` line, and "checkpoint -> ...final.npz".

---
**1/2 — word-prose base (5400 steps)**

Look for `[step 5400]` and `checkpoint -> /content/runs/word_prose/resumed/final.npz`.
If the Colab session disconnects mid-run, re-run this cell — the trainer resumes from the latest checkpoint automatically.

In [0]:
%cd /content/astra
!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 5400 --out /content/runs/word_prose \
  --cache-dir /content/cache 2>&1 | tail -20
print('\n=== prose final ===')
!ls -la /content/runs/word_prose/resumed/final.npz


**2/2 — word-chat fine-tune (3800 steps, warm-started from prose final)**

Look for `[step 3800]` and `checkpoint -> /content/runs/word_chat/resumed/final.npz`.

In [0]:
%cd /content/astra
!python training/gpu_train.py --config configs/astra5m_word_chat.json \
  --steps 3800 --out /content/runs/word_chat \
  --resume /content/runs/word_prose/resumed/final.npz --reset-step \
  --cache-dir /content/cache 2>&1 | tail -20
print('\n=== chat final ===')
!ls -la /content/runs/word_chat/resumed/final.npz


**Download the two checkpoints** to your machine, then place them at:
- `checkpoints/astra5m_word_prose/resumed/final.npz`
- `checkpoints/astra5m_word_chat/resumed/final.npz`

Both are the exact `.npz` format the NumPy inference path loads — no converter needed. Verify locally with `make chat` / `make word-prose` etc.

In [0]:
from google.colab import files
import os
for ckpt in ['/content/runs/word_prose/resumed/final.npz',
             '/content/runs/word_chat/resumed/final.npz']:
    if os.path.exists(ckpt):
        files.download(ckpt)
        print('queued download:', ckpt)
    else:
        print('MISSING:', ckpt)